# Composite FX Alpha: Multi-Factor Daily

| Component | Detail |
|-----------|--------|
| Momentum | Blended 21d/63d log returns |
| Vol Regime | Short/long vol ratio -> weight |
| Vol Scaling | target_vol / realized_vol |
| Drawdown | State machine (NORMAL/REDUCED/FLAT) |
| Sub-portfolios | K=5 Jegadeesh-Titman |
| Rebalancing | Daily target-weight (amount) |

## 1. Setup

In [ ]:
import warnings
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import vectorbtpro as vbt
from numba import njit
from numba.core.errors import NumbaPerformanceWarning
from plotly.subplots import make_subplots

warnings.filterwarnings("ignore", category=NumbaPerformanceWarning)
warnings.filterwarnings(
    "ignore",
    message="Argument at index .* not found in SequenceTaker",
    module=r"vectorbtpro\\.utils\\.chunking",
)

In [ ]:
import multiprocessing
from numba import get_num_threads
from pathlib import Path

n_cores = multiprocessing.cpu_count()
print(f"Cores: {n_cores} | Numba threads: {get_num_threads()}")

def _fullscreen(fig):
    fig.update_layout(width=None, height=None, autosize=True,
        margin=dict(l=30, r=30, t=60, b=30),
        title=dict(font=dict(size=20), x=0.5, xanchor="center"),
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="center", x=0.5))
    return fig

vbt.settings.set("plotting.pre_show_func", _fullscreen)
vbt.settings.returns.year_freq = pd.Timedelta(hours=24) * 252

## 2. Data (Daily)

In [ ]:
_PROJECT_ROOT = Path(__file__).resolve().parent if "__file__" in dir() else Path(".").resolve()
# When run from notebooks/ dir: parent is project root
# When run from project root: "." is project root
# Fallback: search upward for data/
for _p in [Path(".").resolve(), Path(".").resolve().parent, Path(".").resolve().parent.parent]:
    if (_p / "data" / "EUR-USD.parquet").exists():
        _PROJECT_ROOT = _p
        break

def load_fx_data(path="data/EUR-USD.parquet", shift_hours=0):
    resolved = _PROJECT_ROOT / path
    data_raw = vbt.Data.from_parquet(str(resolved))
    symbol = data_raw.symbols[0]
    df = data_raw.data[symbol].set_index("date").sort_index()
    if shift_hours:
        df.index = df.index + pd.Timedelta(hours=shift_hours)
    raw = df.copy()
    raw.columns = [c.lower() for c in raw.columns]
    return raw

raw = load_fx_data(shift_hours=7)
daily = raw.resample("1D").agg({"open":"first","high":"max","low":"min","close":"last"}).dropna()
close = daily["close"].values
log_returns = np.log(daily["close"] / daily["close"].shift(1)).values
print(f"Daily bars: {len(daily):,} | {daily.index[0].date()} -> {daily.index[-1].date()}")

In [ ]:
fig = go.Figure(data=go.Scatter(x=daily.index, y=daily["close"], line=dict(color="#333")))
fig.update_layout(height=400, title="EUR/USD Daily Close")
fig.show()

## 3. Numba Kernels

In [ ]:
@njit(nogil=True)
def momentum_signal_nb(close, w_short, w_long):
    n = len(close); out = np.full(n, np.nan)
    for i in range(w_long, n):
        out[i] = 0.5 * np.log(close[i]/close[i-w_short]) + 0.5 * np.log(close[i]/close[i-w_long])
    return out

@njit(nogil=True)
def regime_weight_nb(vr, low_th, high_th, w_low, w_normal, w_high):
    n = len(vr); out = np.full(n, np.nan)
    for i in range(n):
        if np.isnan(vr[i]): continue
        elif vr[i] < low_th: out[i] = w_low
        elif vr[i] > high_th: out[i] = w_high
        else: out[i] = w_normal
    return out

@njit(nogil=True)
def vol_scaling_nb(ewma_vol, target, cap):
    n = len(ewma_vol); out = np.full(n, np.nan)
    for i in range(n):
        v = ewma_vol[i]
        if not np.isnan(v) and v > 0: out[i] = min(target/v, cap)
    return out

@njit(nogil=True)
def drawdown_control_nb(dd, soft, hard, recovery):
    n = len(dd); mult = np.ones(n); state = 0
    for i in range(n):
        d = dd[i]
        if state==0:
            if d > hard: state=2
            elif d > soft: state=1
        elif state==1:
            if d > hard: state=2
            elif d < recovery: state=0
        elif state==2 and d < recovery: state=0
        if state==1: mult[i]=0.5
        elif state==2: mult[i]=0.0
    return mult

@njit(nogil=True)
def sub_portfolio_weights_nb(direction, regime_wt, vol_scale, dd_mult, n_days, k):
    sub_w = np.zeros((k, n_days))
    for j in range(k):
        current = 0.0
        for i in range(n_days):
            if i >= j*5 and (i-j*5) % (k*5) == 0:
                d=direction[i]; r=regime_wt[i]; v=vol_scale[i]; m=dd_mult[i]
                if not (np.isnan(d) or np.isnan(r) or np.isnan(v) or np.isnan(m)):
                    current = d*r*v*m
            sub_w[j,i] = current
    weights = np.zeros(n_days)
    for i in range(n_days):
        t = 0.0
        for j in range(k): t += sub_w[j,i]
        weights[i] = t/k
    return weights

@njit(nogil=True)
def compute_composite_nb(close, returns, w_short, w_long, vol_short, vol_long,
    ewma_span, target_vol, leverage_cap, vr_low, vr_high,
    mom_w_low, mom_w_normal, mom_w_high, dd_soft, dd_hard, dd_recovery, n_sub):
    n = len(close)
    momentum = momentum_signal_nb(close, w_short, w_long)
    direction = np.full(n, 0.0)
    for i in range(n):
        if not np.isnan(momentum[i]):
            direction[i] = 1.0 if momentum[i]>0 else (-1.0 if momentum[i]<0 else 0.0)
    ss = vbt.generic.nb.rolling_std_1d_nb(returns, vol_short, minp=vol_short, ddof=1)
    sl = vbt.generic.nb.rolling_std_1d_nb(returns, vol_long, minp=vol_long, ddof=1)
    vr = np.full(n, np.nan)
    for i in range(n):
        if not np.isnan(ss[i]) and not np.isnan(sl[i]) and sl[i]>0: vr[i]=ss[i]/sl[i]
    regime_wt = regime_weight_nb(vr, vr_low, vr_high, mom_w_low, mom_w_normal, mom_w_high)
    sq = np.empty(n)
    for i in range(n): sq[i] = returns[i]**2 if not np.isnan(returns[i]) else np.nan
    ev = vbt.generic.nb.ewm_mean_1d_nb(sq, ewma_span, minp=1, adjust=True)
    ewma_vol = np.full(n, np.nan)
    for i in range(n):
        if not np.isnan(ev[i]) and ev[i]>0: ewma_vol[i]=np.sqrt(ev[i])*np.sqrt(252)
    vs = vol_scaling_nb(ewma_vol, target_vol, leverage_cap)
    pe = np.ones(n)
    for i in range(1, n):
        d=direction[i-1]; rw=regime_wt[i-1]; v=vs[i-1]; r=returns[i]
        if not (np.isnan(d) or np.isnan(rw) or np.isnan(v) or np.isnan(r)):
            pe[i]=pe[i-1]*(1+r*d*rw*v)
        else: pe[i]=pe[i-1]
    lb=63; dd=np.zeros(n)
    for i in range(n):
        pk=pe[max(0,i-lb+1)]
        for j in range(max(0,i-lb+1),i+1):
            if pe[j]>pk: pk=pe[j]
        dd[i]=1-pe[i]/pk if pk>0 else 0
    dm = drawdown_control_nb(dd, dd_soft, dd_hard, dd_recovery)
    weights = sub_portfolio_weights_nb(direction, regime_wt, vs, dm, n, n_sub)
    return momentum, direction, vr, regime_wt, ewma_vol, vs, dd, dm, weights

@njit(nogil=True)
def composite_signal_nb(c, target_weights, size_arr):
    tw = vbt.pf_nb.select_nb(c, target_weights)
    if np.isnan(tw): return False, False, False, False
    pos=c.last_position[c.col]; price=c.last_val_price[c.col]; value=c.last_value[c.group]
    if value<=0 or price<=0: return False, False, False, False
    tp = tw*value/price; delta=tp-pos
    if abs(delta*price/value)<0.005: return False, False, False, False
    size_arr[c.i,c.col]=abs(delta)
    if pos>=0 and delta>0: return True, False, False, False
    elif pos>0 and delta<0 and tp>=0: return False, True, False, False
    elif pos>=0 and tp<0: return False, False, True, False
    elif pos<=0 and delta<0: return False, False, True, False
    elif pos<0 and delta>0 and tp<=0: return False, False, False, True
    elif pos<=0 and tp>0: return True, False, False, False
    return False, False, False, False

## 4. Indicators

In [ ]:
COMPOSITE = vbt.IF(
    class_name="CompositeAlpha", short_name="ca",
    input_names=["close", "returns"],
    param_names=["w_short","w_long","vol_short","vol_long","ewma_span","target_vol",
                 "leverage_cap","vr_low","vr_high","mom_w_low","mom_w_normal","mom_w_high",
                 "dd_soft","dd_hard","dd_recovery","n_sub"],
    output_names=["momentum","direction","vol_regime","regime_weight","ewma_vol",
                  "vol_scale","drawdown","dd_multiplier","target_weight"],
).with_apply_func(compute_composite_nb, takes_1d=True,
    w_short=21, w_long=63, vol_short=21, vol_long=252, ewma_span=30,
    target_vol=0.10, leverage_cap=3.0, vr_low=0.8, vr_high=1.2,
    mom_w_low=0.20, mom_w_normal=0.30, mom_w_high=0.50,
    dd_soft=0.12, dd_hard=0.20, dd_recovery=0.10, n_sub=5)
ind = COMPOSITE.run(close, log_returns,
    w_short=21, w_long=63, vol_short=21, vol_long=252, ewma_span=30,
    target_vol=0.10, leverage_cap=3.0, vr_low=0.8, vr_high=1.2,
    mom_w_low=0.20, mom_w_normal=0.30, mom_w_high=0.50,
    dd_soft=0.12, dd_hard=0.20, dd_recovery=0.10, n_sub=5,
    jitted_loop=True, jitted_warmup=True)

### Indicator Inspection

In [ ]:
ind_df = pd.DataFrame({
    "close": daily["close"].values, "momentum": ind.momentum.values,
    "direction": ind.direction.values, "vol_regime": ind.vol_regime.values,
    "regime_wt": ind.regime_weight.values, "vol_scale": ind.vol_scale.values,
    "drawdown": ind.drawdown.values, "dd_mult": ind.dd_multiplier.values,
    "target_wt": ind.target_weight.values,
}, index=daily.index)
print(f"Shape: {ind_df.shape} | NaN: {ind_df.isna().sum().to_dict()}")
print(f"\nTarget weight stats:")
print(f"  mean={ind_df['target_wt'].mean():.4f}, std={ind_df['target_wt'].std():.4f}")
print(f"  min={ind_df['target_wt'].min():.4f}, max={ind_df['target_wt'].max():.4f}")
print(f"  zero%: {(ind_df['target_wt']==0).mean()*100:.1f}%")
ind_df.dropna().tail(15)

## 5. Signal Dashboard

In [ ]:
fig = make_subplots(rows=5, cols=1, shared_xaxes=True, row_heights=[0.25,0.15,0.15,0.2,0.25],
    subplot_titles=("Price","Momentum","Vol Regime","Vol Scale & DD Mult","Target Weight"))
fig.add_trace(go.Scatter(x=daily.index, y=daily["close"], name="Close", line=dict(color="#333")), row=1, col=1)
fig.add_trace(go.Scatter(x=daily.index, y=ind.momentum.values, name="Mom", line=dict(color="#2196F3")), row=2, col=1)
fig.add_hline(y=0, line_color="gray", row=2, col=1)
fig.add_trace(go.Scatter(x=daily.index, y=ind.vol_regime.values, name="VR", line=dict(color="#FF9800")), row=3, col=1)
fig.add_hline(y=0.8, line_dash="dot", line_color="green", row=3, col=1)
fig.add_hline(y=1.2, line_dash="dot", line_color="red", row=3, col=1)
fig.add_trace(go.Scatter(x=daily.index, y=ind.vol_scale.values, name="Vol Scale", line=dict(color="#4CAF50")), row=4, col=1)
fig.add_trace(go.Scatter(x=daily.index, y=ind.dd_multiplier.values, name="DD Mult", line=dict(color="#F44336", dash="dash")), row=4, col=1)
fig.add_trace(go.Scatter(x=daily.index, y=ind.target_weight.values, name="Target Wt", fill="tozeroy", line=dict(color="#9C27B0")), row=5, col=1)
fig.update_layout(height=1200, title="Composite Alpha Signal Dashboard")
fig.show()

In [ ]:
fig = make_subplots(rows=2, cols=1, shared_xaxes=True, subplot_titles=("Drawdown %","DD Multiplier"))
fig.add_trace(go.Scatter(x=daily.index, y=ind.drawdown.values*100, fill="tozeroy", line=dict(color="#F44336"), name="DD%"), row=1, col=1)
fig.add_hline(y=12, line_dash="dot", line_color="orange", annotation_text="Soft", row=1, col=1)
fig.add_hline(y=20, line_dash="dot", line_color="red", annotation_text="Hard", row=1, col=1)
fig.add_trace(go.Scatter(x=daily.index, y=ind.dd_multiplier.values, fill="tozeroy", line=dict(color="#2196F3"), name="Mult"), row=2, col=1)
fig.update_layout(height=500, title="Drawdown Control State Machine")
fig.show()

## 6. Backtest

In [ ]:
pf = vbt.Portfolio.from_signals(
    close=daily["close"],
    signal_func_nb=composite_signal_nb,
    signal_args=(vbt.Rep("tw"), vbt.Rep("sz")),
    broadcast_named_args=dict(tw=ind.target_weight.values,
        sz=vbt.RepEval("np.full(wrapper.shape_2d, np.nan)")),
    size=vbt.RepEval("np.full(wrapper.shape_2d, np.nan)"),
    size_type="amount", accumulate=True, upon_opposite_entry="Reverse",
    leverage=2.0, leverage_mode="lazy", fees=0.00035, init_cash=1_000_000, freq="1D")
print(f"Trades: {pf.trades.count()}")

## 7. Results

In [ ]:
print("PORTFOLIO STATS\n" + "="*50)
print(pf.stats().to_string())

In [ ]:
fig = pf.plot(subplots=["cumulative_returns", "drawdowns", "underwater", "trade_pnl"])
fig.update_layout(height=900, title="Composite FX Alpha: Summary")
fig.show()

In [ ]:
if pf.trades.count() > 0:
    fig = make_subplots(rows=2, cols=2,
        subplot_titles=("Trade PnL (%)", "MAE", "MFE", "Running Edge Ratio"))
    pf.trades.plot_pnl(pct_scale=True, fig=fig, add_trace_kwargs=dict(row=1, col=1))
    pf.trades.plot_mae(fig=fig, add_trace_kwargs=dict(row=1, col=2))
    pf.trades.plot_mfe(fig=fig, add_trace_kwargs=dict(row=2, col=1))
    pf.trades.plot_running_edge_ratio(fig=fig, add_trace_kwargs=dict(row=2, col=2))
    fig.update_layout(height=800, showlegend=False)
    fig.show()

In [ ]:
rets = pf.returns
monthly = rets.resample("ME").apply(lambda x: (1+x).prod()-1)
df_m = pd.DataFrame({"return": monthly})
df_m["year"] = df_m.index.year; df_m["month"] = df_m.index.month
pivot = df_m.pivot_table(values="return", index="year", columns="month", aggfunc="first")
mn = ["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"]
pivot.columns = mn[:len(pivot.columns)]
fig = go.Figure(data=go.Heatmap(z=pivot.values*100, x=pivot.columns.tolist(),
    y=[str(y) for y in pivot.index], colorscale="RdYlGn", texttemplate="%{z:.1f}%", zmid=0))
fig.update_layout(title="Monthly Returns (%)", height=300+len(pivot)*30)
fig.show()

## 8. Target Vol Sweep

In [ ]:
results = {}
for tv in [0.05, 0.08, 0.10, 0.15]:
    ind_i = COMPOSITE.run(close, log_returns,
        w_short=21, w_long=63, vol_short=21, vol_long=252, ewma_span=30,
        target_vol=tv, leverage_cap=3.0, vr_low=0.8, vr_high=1.2,
        mom_w_low=0.20, mom_w_normal=0.30, mom_w_high=0.50,
        dd_soft=0.12, dd_hard=0.20, dd_recovery=0.10, n_sub=5,
        jitted_loop=True, jitted_warmup=True)
    pf_i = vbt.Portfolio.from_signals(
        close=daily["close"], signal_func_nb=composite_signal_nb,
        signal_args=(vbt.Rep("tw"), vbt.Rep("sz")),
        broadcast_named_args=dict(tw=ind_i.target_weight.values,
            sz=vbt.RepEval("np.full(wrapper.shape_2d, np.nan)")),
        size=vbt.RepEval("np.full(wrapper.shape_2d, np.nan)"),
        size_type="amount", accumulate=True, upon_opposite_entry="Reverse",
        leverage=2.0, leverage_mode="lazy", fees=0.00035, init_cash=1_000_000, freq="1D")
    results[f"TV={tv}"] = pf_i.stats()
print(pd.DataFrame(results).to_string())

In [ ]:
fig = go.Figure()
for tv in [0.05, 0.08, 0.10, 0.15]:
    ind_i = COMPOSITE.run(close, log_returns, w_short=21, w_long=63, vol_short=21, vol_long=252,
        ewma_span=30, target_vol=tv, leverage_cap=3.0, vr_low=0.8, vr_high=1.2,
        mom_w_low=0.20, mom_w_normal=0.30, mom_w_high=0.50,
        dd_soft=0.12, dd_hard=0.20, dd_recovery=0.10, n_sub=5,
        jitted_loop=True, jitted_warmup=True)
    pf_i = vbt.Portfolio.from_signals(
        close=daily["close"], signal_func_nb=composite_signal_nb,
        signal_args=(vbt.Rep("tw"), vbt.Rep("sz")),
        broadcast_named_args=dict(tw=ind_i.target_weight.values,
            sz=vbt.RepEval("np.full(wrapper.shape_2d, np.nan)")),
        size=vbt.RepEval("np.full(wrapper.shape_2d, np.nan)"),
        size_type="amount", accumulate=True, upon_opposite_entry="Reverse",
        leverage=2.0, leverage_mode="lazy", fees=0.00035, init_cash=1_000_000, freq="1D")
    fig.add_trace(go.Scatter(x=daily.index, y=(1+pf_i.returns).cumprod().values, name=f"TV={tv}"))
fig.update_layout(height=500, title="Equity Curves by Target Vol")
fig.show()